[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/03_quality_filtering.ipynb)

# Step 3 — Quality Filtering and LLM-as-Judge

Filter synthetic Q&A with cheap heuristics, then score survivors with a judge model.

## Learning objectives
- Apply deduplication, format checks, and prompt-leakage detection
- Score samples on correctness, coherence, instruction-following, and plausibility
- Optionally compare candidates pairwise against seed examples

In [ ]:
from pathlib import Path
import os
from aieng.syn_data.text import (
    DEFAULT_JUDGE_THRESHOLD,
    RESULTS_DIR,
    SYNTHETIC_FILTERED_PATH,
    SYNTHETIC_RAW_PATH,
    QASample,
    apply_heuristic_filters,
    create_judge_client,
    filter_with_judge,
    load_typed_jsonl,
    save_typed_jsonl,
    summarize_judge_scores,
    use_repo_root,
    write_json,
)
from dotenv import load_dotenv
from rich.console import Console
from rich.table import Table

# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_dotenv()
use_repo_root(Path("."))

console = Console(width=100)

The notebook path has been set to: /home/coder/synthetic-data-bootcamp


In [2]:
# TODO: remove this before merging into main
%load_ext autoreload
%autoreload 2

## 1. Heuristic filtering

In [3]:
raw_samples = load_typed_jsonl(SYNTHETIC_RAW_PATH, QASample.from_dict)
kept, rejected = apply_heuristic_filters(raw_samples)
print(f"Kept {len(kept)} / {len(raw_samples)} samples")

Kept 136 / 144 samples


## 2. LLM-as-judge absolute scoring

Here the judge model is evaluating quality of the generated data by our teacher model after applying basic heuristic filtering. The minimum pass score is defined in configs. 

In [ ]:
# TODO: change judge model to another family of models (e.g claude or gpt-4o)
judge = create_judge_client()

filtered_samples, judge_scores, rejected = filter_with_judge(
    judge,
    kept,
    threshold=DEFAULT_JUDGE_THRESHOLD,
)
console.print(
    f"[bold green]After judge filter:[/bold green] [yellow]{len(filtered_samples)}[/yellow] kept, "
    f"[red]{len(rejected)}[/red] rejected [dim](heuristics + judge)[/dim]"
)
summarize_judge_scores(judge_scores)

2026-06-25 22:04:05,635 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
    "correctness": 5,
    "coherence": 5,
    "instruction_following": 5,
    "factual_plausibility": 5,
    "reasoning": "The model answer is perfectly correct and matches the reference answer word-for-word."
}
*********** End of JSON payload ***********
2026-06-25 22:04:07,039 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
    "correctness": 5,
    "coherence": 5,
    "instruction_following": 5,
    "factual_plausibility": 5,
    "reasoning": "The model answer is a perfect match to the reference answer, providing the exact definition requested."
}
*********** End of JSON payload ***********
2026-06-25 22:04:14,193 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
    "correctness": 5,
    "coherence": 5,
    "instruction_following": 5,
    "factual_plausibility": 5,
    "reasoning": "The model answer i

After judge filter: 136 kept, 0 rejected (heuristics + judge)

{'correctness': 5.0,
 'coherence': 5.0,
 'instruction_following': 5.0,
 'factual_plausibility': 5.0,
 'average': 5.0}

## 3. Save filtered corpus and quality report

In [5]:
save_typed_jsonl(
    SYNTHETIC_FILTERED_PATH,
    filtered_samples,
    to_dict=QASample.to_dict,
)

quality_report = {
    "input_count": len(raw_samples),
    "after_heuristics": len(kept),
    "after_judge": len(filtered_samples),
    "judge_threshold": DEFAULT_JUDGE_THRESHOLD,
    "judge_summary": summarize_judge_scores(judge_scores),
    "rejected": rejected,
}
write_json(RESULTS_DIR / "quality_report.json", quality_report)
from rich import box

table = Table(title="Quality Report", box=box.ROUNDED)
table.add_column("Metric", style="bold cyan")
table.add_column("Value", style="bold yellow")

for k, v in quality_report.items():
    if isinstance(v, dict):
        # If value is a dictionary, show sub-keys and values
        for subk, subv in v.items():
            table.add_row(f"{k}.{subk}", str(subv))
    elif isinstance(v, list):
        table.add_row(k, f"{len(v)} items")
    else:
        table.add_row(k, str(v))
console.print(table)

                 Quality Report                  
╭─────────────────────────────────────┬─────────╮
│ Metric                              │ Value   │
├─────────────────────────────────────┼─────────┤
│ input_count                         │ 144     │
│ after_heuristics                    │ 136     │
│ after_judge                         │ 136     │
│ judge_threshold                     │ 3.5     │
│ judge_summary.correctness           │ 5.0     │
│ judge_summary.coherence             │ 5.0     │
│ judge_summary.instruction_following │ 5.0     │
│ judge_summary.factual_plausibility  │ 5.0     │
│ judge_summary.average               │ 5.0     │
│ rejected                            │ 0 items │
╰─────────────────────────────────────┴─────────╯